In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>
span {
    font-size: 24px;
}
</style>
"""))

In [2]:
RANDOM_STATE = 42 # for reproducibility

In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("data/hedging_dataset")
EXCLUDED_COMPANIES = {"ASML", "MU"}

csv_files = sorted(
    path for path in BASE_DIR.rglob("*.csv")
    if path.relative_to(BASE_DIR).parts[0] not in EXCLUDED_COMPANIES
)
if not csv_files:
    raise FileNotFoundError(f"No CSV files found under: {BASE_DIR}")

dfs = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)

    # Validate schema
    required_cols = {"sentence", "isHedge"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {missing}")

    # Keep only the two required columns
    df = df[["sentence", "isHedge"]].copy()

    # Validate label values (accept 0, 1, 0.0, 1.0)
    labels = pd.to_numeric(df["isHedge"], errors="coerce")
    invalid_mask = ~labels.isin([0, 1]) & labels.notna()
    if invalid_mask.any():
        bad_vals = df.loc[invalid_mask, "isHedge"].unique().tolist()
        raise ValueError(f"{csv_path} has non-binary isHedge values: {bad_vals}")

    # Normalize to integer 0/1
    df["isHedge"] = labels.astype("Int64")

    dfs.append(df)

train = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(csv_files)} files")
print(f"Combined rows: {len(train)}")
display(train.head())

Loaded 134 files
Combined rows: 46094


,sentence,isHedge
0,Thank you.,0
1,"Good afternoon, and thanks to everyone for joi...",0
2,"Speaking first is Apple's CEO, Tim Cook, and h...",0
3,"After that, we'll open the call to questions f...",0
4,Please note that some of the information you'l...,1


In [4]:
print(train['isHedge'].value_counts())

isHedge
0    35075
1    11019
Name: count, dtype: Int64


<h3>Feature Engineering: TF-IDF Vectorization</h3>
<span>
<table>
    <tr>
        <td><strong>Parameter</strong></td>
        <td><strong>Description</strong></td>
    </tr>
    <tr>
        <td><code>max_features</code></td>
        <td>Only the top <code>max_features</code> tokens by total TF-IDF score are kept in the TF-IDF matrix with the rest being <strong>discarded</strong>.</td>
    </tr>
    <tr>
        <td>N-gram range</td>
        <td>A selection of N words. A 2-gram, known as a bigram, is a group of two words. Bigrams relevant to us are those such as "We believe", "I expect". In this project, we will use the <code>(1, 2)</code> range which includes unigrams (single words) and bigrams.</td>
    </tr>
    <tr>
        <td><code>min_df</code> (minimum document frequency)</td>
        <td>The minimum number of sentences a word/token needs to appear in for it to be included within the TF-IDF vocabulary.</td>
    </tr>
    <tr>
        <td>Sublinear TF scaling</td>
        <td>A log transform applied to make the jump from 1 to 2 occurrences meaningful but from 8 to 9 less so. TF becomes <code>1+log(count)</code> where <code>count</code> is the <strong>raw count of the token</strong>.</td>
    </tr>
</table>
</span>

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(train["sentence"])
y = pd.to_numeric(train["isHedge"], errors="raise").astype("int8")

if not y.isin([0, 1]).all():
    raise ValueError("train['isHedge'] must only contain binary labels 0 or 1.")

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("y distribution:")
print(y.value_counts(dropna=False).sort_index())

vectorizer.get_feature_names_out()

X shape: (46094, 27737)
y shape: (46094,)
y distribution:
isHedge
0    35075
1    11019
Name: count, dtype: int64


array(['aaa', 'aaa titles', 'aaron', ..., 'zip', 'zip codes', 'zones'],
      shape=(27737,), dtype=object)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

classifier = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=1000)
classifier.fit(X, y)

test_csv_files = sorted(
    path for path in BASE_DIR.rglob("*.csv")
    if path.relative_to(BASE_DIR).parts[0] in EXCLUDED_COMPANIES
)
if not test_csv_files:
    raise FileNotFoundError(
        f"No CSV files found for EXCLUDED_COMPANIES: {sorted(EXCLUDED_COMPANIES)}"
    )

X_test = pd.concat(
    [pd.read_csv(path)[["sentence", "isHedge"]].copy() for path in test_csv_files],
    ignore_index=True,
)
X_test["sentence"] = X_test["sentence"].astype(str)
y_test = X_test.pop('isHedge')
y_test = pd.to_numeric(y_test, errors="coerce")

valid_y_test = y_test.isin([0, 1])
if not valid_y_test.all():
    dropped = int((~valid_y_test).sum())
    print(f"Dropping {dropped} test rows with invalid isHedge labels before evaluation.")
    X_test = X_test.loc[valid_y_test].reset_index(drop=True)
    y_test = y_test.loc[valid_y_test]

y_test = y_test.astype("int8")

X_test_tfidf = vectorizer.transform(X_test["sentence"])
y_pred = classifier.predict(X_test_tfidf)

assert len(y_pred) == len(y_test)
print(classification_report(y_test, y_pred, digits=4))


              precision    recall  f1-score   support

           0     0.9125    0.8647    0.8880      7969
           1     0.7679    0.8437    0.8040      4228

    accuracy                         0.8574     12197
   macro avg     0.8402    0.8542    0.8460     12197
weighted avg     0.8624    0.8574    0.8589     12197

